# 03 — Sentiment + Risk Aggregation
Scores each article with VADER (fast) or RoBERTa (accurate), then builds daily topic/risk time series.
Input: `data/classified_articles.csv`
Outputs: `data/scored_articles.csv`, `data/timeseries.csv`, `data/timeseries_wide.csv`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/classified_articles.csv', parse_dates=['date'])

# Backward-compatible defaults in case older classified files are loaded.
if 'primary_topic' not in df.columns:
    df['primary_topic'] = 'other'
if 'risk_score' not in df.columns:
    df['risk_score'] = 0

df['risk_score'] = pd.to_numeric(df['risk_score'], errors='coerce').fillna(0).astype(int)
print(df.shape)
print(df.columns.tolist())

(103343, 14)
['id', 'date', 'section', 'headline', 'body', 'wordcount', 'url', 'subtopic', 'topics', 'primary_topic', 'countries', 'organizations', 'commodities', 'risk_score']


## Option A — VADER (recommended for this project)
Fast, no GPU, designed for news/social media. Outputs compound score in [-1, 1].

In [3]:
# pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_score(text: str) -> float:
    # Use headline + first 300 chars of body — VADER degrades on very long text
    snippet = str(text)[:300]
    return sia.polarity_scores(snippet)['compound']


df['text_sample'] = df['headline'].fillna('') + '. ' + df['body'].fillna('').str[:300]
df['sentiment'] = df['text_sample'].apply(vader_score)

# Map compound to label
df['sentiment_label'] = pd.cut(
    df['sentiment'],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=['negative', 'neutral', 'positive']
)

# Composite risk index combines severity and sentiment intensity.
# Higher when risk terms are present AND tone is negative/intense.
df['sentiment_intensity'] = df['sentiment'].abs()
df['risk_index'] = (
    0.7 * df['risk_score'].clip(lower=0, upper=10) / 10.0
    + 0.3 * df['sentiment_intensity']
).round(4)

print(df['sentiment_label'].value_counts())
df[['headline', 'primary_topic', 'risk_score', 'sentiment', 'risk_index']].head(5)

sentiment_label
negative    61617
positive    35306
neutral      6420
Name: count, dtype: int64


,headline,primary_topic,risk_score,sentiment,risk_index
0,Middle East crisis pushes up oil prices – and ...,ai,0,-0.6293,0.1888
1,Middle East crisis pushes up oil prices – and ...,ai,0,-0.6293,0.1888
2,Middle East crisis pushes up oil prices – and ...,ai,0,-0.6293,0.1888
3,Will shipping in the strait of Hormuz – and oi...,geopolitics_conflict,0,-0.3400,0.1020
4,Will shipping in the strait of Hormuz – and oi...,geopolitics_conflict,0,-0.3400,0.1020


## Option B — RoBERTa (better accuracy, ~30 min on CPU for 10k)
Use `cardiffnlp/twitter-roberta-base-sentiment-latest` — trained on social/news text.

In [ ]:
# from transformers import pipeline
# import torch
#
# device = 0 if torch.cuda.is_available() else -1
# sentiment_pipe = pipeline(
#     'sentiment-analysis',
#     model='cardiffnlp/twitter-roberta-base-sentiment-latest',
#     device=device,
#     truncation=True,
#     max_length=512
# )

#
# # Batch inference for speed
# texts = (df['headline'] + '. ' + df['body'].str[:300]).tolist()
# results = sentiment_pipe(texts, batch_size=32)
#
# label_map = {'positive': 1, 'neutral': 0, 'negative': -1}
# df['sentiment_label'] = [r['label'].lower() for r in results]
# df['sentiment'] = [label_map[r['label'].lower()] * r['score'] for r in results]

In [5]:
# Save article-level scored data
if 'text_sample' in df.columns:
    df.drop(columns=['text_sample'], inplace=True)

df.to_csv('data/scored_articles.csv', index=False)
print('Saved: data/scored_articles.csv')

# Build long-format daily time series by primary topic
sent_neg = (df['sentiment_label'] == 'negative').astype(int)

timeseries = (
    df.assign(date=df['date'].dt.date, neg_flag=sent_neg)
      .groupby(['date', 'primary_topic'], as_index=False)
      .agg(
          article_count=('id', 'count'),
          avg_sentiment=('sentiment', 'mean'),
          avg_risk_score=('risk_score', 'mean'),
          avg_risk_index=('risk_index', 'mean'),
          high_risk_share=('risk_score', lambda s: float((s >= 7).mean())),
          negative_share=('neg_flag', 'mean'),
      )
)

timeseries.to_csv('data/timeseries.csv', index=False)
print('Saved: data/timeseries.csv')

# Build wide-format daily matrix for forecasting
timeseries_wide = (
    timeseries
    .pivot_table(index='date', columns='primary_topic', values='article_count', fill_value=0)
    .reset_index()
)
timeseries_wide.columns = [str(c) for c in timeseries_wide.columns]
timeseries_wide.to_csv('data/timeseries_wide.csv', index=False)
print('Saved: data/timeseries_wide.csv')

print('\nScored articles describe:')
print(df[['wordcount', 'sentiment', 'risk_score', 'risk_index']].describe())

print('\nTimeseries sample:')
print(timeseries.head(10).to_string(index=False))

Saved: data/scored_articles.csv
Saved: data/timeseries.csv
Saved: data/timeseries_wide.csv

Scored articles describe:
           wordcount      sentiment  risk_score     risk_index
count  103343.000000  103343.000000    103343.0  103343.000000
mean     1553.561644      -0.195901         0.0       0.164701
std      2394.998835       0.590147         0.0       0.087593
min       100.000000      -0.988600         0.0       0.000000
25%       613.000000      -0.750600         0.0       0.095500
50%       824.000000      -0.296000         0.0       0.179800
75%      1150.000000       0.318200         0.0       0.242200
max     38962.000000       0.988800         0.0       0.296600

Timeseries sample:
      date        primary_topic  article_count  avg_sentiment  avg_risk_score  avg_risk_index  high_risk_share  negative_share
2021-10-28 geopolitics_conflict              3         0.0772             0.0         0.02320              0.0             0.0
2021-11-26 geopolitics_conflict          